In [ ]:
"""
analisis_muestras.py
--------------------
Sesión práctica: corrección, exploración y análisis de un archivo plano.

Flujo:
    1. Carga y primera inspección
    2. Corrección de datos (limpieza con criterio biológico)
    3. Exploración (descriptiva, unicidad, distribuciones)
    4. Análisis (por especie, por sitio, relaciones)
    5. Resultados (tablas y figuras exportadas)

Salidas:
    - muestras_clean.csv          (dataset corregido)
    - resumen_especies.csv        (tabla resumen)
    - resumen_sitios.csv          (tabla resumen)
    - fig_*.png                   (figuras)
    - log_limpieza.txt            (bitácora de cambios)

Requisitos:
    pip install pandas matplotlib
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Semilla para reproducibilidad (no afecta aquí, pero es buena práctica)
RANDOM_STATE = 42

# Carpeta de salida (todo lo que generemos va aquí)
OUT = "resultados"
os.makedirs(OUT, exist_ok=True)

# Bitácora: registramos cada decisión de limpieza
LOG_PATH = os.path.join(OUT, "log_limpieza.txt")
log_lines = []

def log(msg):
    """Escribe un mensaje en la bitácora y en pantalla."""
    print(msg)
    log_lines.append(msg)


# ======================================================================
# 1. CARGA Y PRIMERA INSPECCIÓN
# ======================================================================
# Objetivo: saber con qué estamos trabajando antes de tocarlo.
# Regla de oro: NUNCA modificar el archivo original.

log("=" * 60)
log("1. CARGA Y PRIMERA INSPECCIÓN")
log("=" * 60)

df_raw = pd.read_csv("muestras.csv")

log(f"Filas:    {len(df_raw)}")
log(f"Columnas: {list(df_raw.columns)}")
log(f"\nPrimeras filas:\n{df_raw.head()}")

# Tipos y valores faltantes
log("\nTipos de datos:")
log(str(df_raw.dtypes))

log("\nValores faltantes por columna:")
log(str(df_raw.isna().sum()))

# Nota de clase: hasta aquí NO hemos corregido nada. Solo miramos.


# ======================================================================
# 2. CORRECCIÓN DE DATOS
# ======================================================================
# Cada paso responde a un problema concreto. Los hacemos uno por uno
# para que se vea el efecto de cada decisión.

log("\n" + "=" * 60)
log("2. CORRECCIÓN DE DATOS")
log("=" * 60)

df = df_raw.copy()   # trabajamos sobre una copia


# ----------------------------------------------------------------------
# 2.1 Duplicados exactos
# ----------------------------------------------------------------------
# Si dos filas son idénticas en TODAS las columnas, probablemente
# es un error de doble registro (doble clic, doble export, etc.).

dup_mask = df.duplicated(keep="first")
n_dup = dup_mask.sum()
log(f"\n[2.1] Duplicados exactos encontrados: {n_dup}")

if n_dup > 0:
    log("Filas duplicadas (se eliminarán):")
    log(str(df[dup_mask]))
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    log(f"→ Dataset tras eliminar duplicados: {len(df)} filas")


# ----------------------------------------------------------------------
# 2.2 Nombres de sitio inconsistentes
# ----------------------------------------------------------------------
# Problema típico: "Reserva El Roble" vs "Reserva el Roble".
# Solución: normalizar (minúsculas, sin espacios extra) y mapear
# al nombre oficial.

log("\n[2.2] Nombres de sitio únicos (antes de normalizar):")
for s in sorted(df["sitio"].unique()):
    log(f"    - '{s}'")


def normalizar_sitio(nombre: str) -> str:
    """Limpia un nombre de sitio: quita espacios y baja a minúsculas."""
    if pd.isna(nombre):
        return nombre
    return " ".join(str(nombre).strip().lower().split())


df["sitio_norm"] = df["sitio"].apply(normalizar_sitio)

log("\nNombres de sitio únicos (después de normalizar):")
for s in sorted(df["sitio_norm"].unique()):
    log(f"    - '{s}'")

# Mapeo al nombre oficial (el que queremos en la base limpia)
MAPEO_SITIOS = {
    "reserva el roble": "Reserva El Roble",
    "parque nacional la campana": "Parque Nacional La Campana",
    "estación biológica senda darwin": "Estación Biológica Senda Darwin",
    "reserva costera valdiviana": "Reserva Costera Valdiviana",
    "humedal río cruces": "Humedal Río Cruces",
    "bosque pehuén": "Bosque Pehuén",
}

df["sitio"] = df["sitio_norm"].map(MAPEO_SITIOS)

# ¿Quedó algún sitio sin mapear? Eso es un problema que hay que ver.
sin_mapear = df["sitio"].isna().sum()
if sin_mapear > 0:
    log(f"\n⚠ ATENCIÓN: {sin_mapear} filas con sitio no reconocido:")
    log(str(df[df["sitio"].isna()][["id_muestra", "sitio_norm"]]))
    log("→ Revisar manualmente. Por ahora se conservan con el nombre normalizado.")
    df["sitio"] = df["sitio"].fillna(df["sitio_norm"])

df = df.drop(columns=["sitio_norm"])
log("→ Sitios estandarizados.")


# ----------------------------------------------------------------------
# 2.3 Coordenadas inconsistentes por sitio
# ----------------------------------------------------------------------
# Un mismo sitio debería tener SIEMPRE las mismas coordenadas.
# Si no, hay un error de tipeo en alguna fila.
# Estrategia: quedarnos con la MEDIANA por sitio (robusta a outliers).

log("\n[2.3] Coordenadas únicas por sitio (antes de corregir):")
coord_check = df.groupby("sitio")[["latitud", "longitud"]].nunique()
log(str(coord_check))

# Detectar sitios con más de una coordenada distinta
sitios_inconsistentes = coord_check[(coord_check["latitud"] > 1) |
                                    (coord_check["longitud"] > 1)].index.tolist()

if sitios_inconsistentes:
    log(f"\nSitios con coordenadas inconsistentes: {sitios_inconsistentes}")
    for s in sitios_inconsistentes:
        log(f"\n  Detalle de '{s}':")
        log(str(df[df["sitio"] == s][["id_muestra", "latitud", "longitud"]]))

    # Reemplazamos por la mediana del sitio
    medianas = df.groupby("sitio")[["latitud", "longitud"]].median()
    for s in sitios_inconsistentes:
        df.loc[df["sitio"] == s, "latitud"]  = medianas.loc[s, "latitud"]
        df.loc[df["sitio"] == s, "longitud"] = medianas.loc[s, "longitud"]
    log("\n→ Coordenadas reemplazadas por la mediana del sitio.")
else:
    log("\n→ No se detectaron inconsistencias de coordenadas.")


# ----------------------------------------------------------------------
# 2.4 Validación de tipos y rangos biológicos
# ----------------------------------------------------------------------
# peso_g y longitud_cm deben ser numéricos y positivos.
# Si hay valores absurdos (peso negativo, longitud = 0), los marcamos.

log("\n[2.4] Validación de rangos biológicos")

for col in ["peso_g", "longitud_cm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

invalidos = df[(df["peso_g"] <= 0) | (df["longitud_cm"] <= 0)]
log(f"Filas con peso o longitud no positivos: {len(invalidos)}")
if len(invalidos) > 0:
    log(str(invalidos[["id_muestra", "peso_g", "longitud_cm"]]))
    df = df[(df["peso_g"] > 0) & (df["longitud_cm"] > 0)].reset_index(drop=True)
    log(f"→ Dataset tras eliminar valores inválidos: {len(df)} filas")


# ----------------------------------------------------------------------
# 2.5 Valores faltantes
# ----------------------------------------------------------------------
# No siempre hay que eliminar. Depende de cuántos sean y de qué columna.

log("\n[2.5] Valores faltantes después de la limpieza:")
log(str(df.isna().sum()))

n_antes = len(df)
df = df.dropna(subset=["peso_g", "longitud_cm", "sitio", "especie"]).reset_index(drop=True)
log(f"→ Filas eliminadas por faltantes en columnas clave: {n_antes - len(df)}")
log(f"→ Dataset final de limpieza: {len(df)} filas")


# ----------------------------------------------------------------------
# 2.6 Resumen de la limpieza
# ----------------------------------------------------------------------

log("\n[2.6] Resumen de limpieza")
log(f"    Filas originales:   {len(df_raw)}")
log(f"    Filas finales:      {len(df)}")
log(f"    Filas eliminadas:   {len(df_raw) - len(df)}")
log(f"    Sitios únicos:      {df['sitio'].nunique()}")
log(f"    Especies únicas:    {df['especie'].nunique()}")


# ======================================================================
# 3. EXPLORACIÓN
# ======================================================================
# Objetivo: entender la estructura y distribución de los datos limpios.

log("\n" + "=" * 60)
log("3. EXPLORACIÓN")
log("=" * 60)

# 3.1 Descriptiva general
log("\n[3.1] Estadística descriptiva general:")
log(str(df[["peso_g", "longitud_cm"]].describe().round(2)))

# 3.2 Muestras por sitio
log("\n[3.2] Muestras por sitio:")
log(str(df["sitio"].value_counts()))

# 3.3 Muestras por especie
log("\n[3.3] Muestras por especie:")
log(str(df["especie"].value_counts()))

# 3.4 Rango temporal
log("\n[3.4] Rango temporal del muestreo:")
fechas = pd.to_datetime(df["fecha"])
log(f"    Desde: {fechas.min().date()}")
log(f"    Hasta: {fechas.max().date()}")
log(f"    Días únicos de muestreo: {fechas.dt.date.nunique()}")

# 3.5 Tabla cruzada especie × sitio
log("\n[3.5] Tabla cruzada especie × sitio (n muestras):")
crosstab = pd.crosstab(df["especie"], df["sitio"])
log(str(crosstab))


# ======================================================================
# 4. ANÁLISIS
# ======================================================================
# Objetivo: responder preguntas biológicas con los datos corregidos.

log("\n" + "=" * 60)
log("4. ANÁLISIS")
log("=" * 60)

# 4.1 Resumen por especie
resumen_especies = (
    df.groupby("especie")
      .agg(
          n_muestras   = ("id_muestra", "count"),
          peso_medio   = ("peso_g", "mean"),
          peso_sd      = ("peso_g", "std"),
          peso_min     = ("peso_g", "min"),
          peso_max     = ("peso_g", "max"),
          largo_medio  = ("longitud_cm", "mean"),
          largo_sd     = ("longitud_cm", "std"),
      )
      .round(2)
      .sort_values("n_muestras", ascending=False)
)
log("\n[4.1] Resumen por especie:")
log(str(resumen_especies))

# 4.2 Resumen por sitio
resumen_sitios = (
    df.groupby("sitio")
      .agg(
          n_muestras    = ("id_muestra", "count"),
          n_especies    = ("especie", "nunique"),
          peso_medio    = ("peso_g", "mean"),
          largo_medio   = ("longitud_cm", "mean"),
      )
      .round(2)
      .sort_values("n_muestras", ascending=False)
)
log("\n[4.2] Resumen por sitio:")
log(str(resumen_sitios))

# 4.3 Relación peso–longitud (por especie)
log("\n[4.3] Correlación peso–longitud por especie:")
correlaciones = (
    df.groupby("especie")
      .apply(lambda g: g["peso_g"].corr(g["longitud_cm"]))
      .round(3)
      .sort_values(ascending=False)
)
log(str(correlaciones))

# 4.4 Especies por sitio (riqueza)
riqueza = df.groupby("sitio")["especie"].nunique().sort_values(ascending=False)
log("\n[4.4] Riqueza específica por sitio:")
log(str(riqueza))


# ======================================================================
# 5. RESULTADOS
# ======================================================================
# Objetivo: exportar todo lo que puede ir a un informe o paper.

log("\n" + "=" * 60)
log("5. RESULTADOS")
log("=" * 60)

# 5.1 Exportar dataset limpio
clean_path = os.path.join(OUT, "muestras_clean.csv")
df.to_csv(clean_path, index=False, encoding="utf-8")
log(f"\n[5.1] Dataset limpio exportado: {clean_path}")

# 5.2 Exportar tablas resumen
resumen_especies.to_csv(os.path.join(OUT, "resumen_especies.csv"))
resumen_sitios.to_csv(os.path.join(OUT, "resumen_sitios.csv"))
log(f"[5.2] Tablas resumen exportadas en '{OUT}/'")

# 5.3 Figuras
plt.style.use("seaborn-v0_8-whitegrid")

# Figura 1: muestras por sitio
fig, ax = plt.subplots(figsize=(9, 5))
resumen_sitios["n_muestras"].sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Número de muestras")
ax.set_ylabel("")
ax.set_title("Muestras por sitio")
plt.tight_layout()
fig.savefig(os.path.join(OUT, "fig_muestras_por_sitio.png"), dpi=150)
plt.close(fig)

# Figura 2: distribución de peso por especie
fig, ax = plt.subplots(figsize=(10, 5))
df.boxplot(column="peso_g", by="especie", ax=ax, rot=45)
ax.set_title("Distribución de peso por especie")
ax.set_xlabel("")
ax.set_ylabel("Peso (g)")
plt.suptitle("")   # quitar el título automático de pandas
plt.tight_layout()
fig.savefig(os.path.join(OUT, "fig_peso_por_especie.png"), dpi=150)
plt.close(fig)

# Figura 3: relación peso–longitud, coloreado por especie
fig, ax = plt.subplots(figsize=(8, 6))
for esp, grupo in df.groupby("especie"):
    ax.scatter(grupo["longitud_cm"], grupo["peso_g"], label=esp, alpha=0.7, s=30)
ax.set_xlabel("Longitud (cm)")
ax.set_ylabel("Peso (g)")
ax.set_title("Relación peso–longitud por especie")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
fig.savefig(os.path.join(OUT, "fig_peso_vs_longitud.png"), dpi=150)
plt.close(fig)

log(f"[5.3] Figuras exportadas en '{OUT}/'")

# 5.4 Guardar bitácora
with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write(f"Bitácora de limpieza — {datetime.now():%Y-%m-%d %H:%M}\n")
    f.write("=" * 60 + "\n")
    f.write("\n".join(log_lines))

log(f"[5.4] Bitácora guardada: {LOG_PATH}")

log("\n" + "=" * 60)
log("FIN DEL ANÁLISIS")
log("=" * 60)
log(f"\nRevisa la carpeta '{OUT}/' para ver todos los resultados.")